<a href="https://colab.research.google.com/github/sookhyangkim/research/blob/main/papers/2026-xiaofu-xingtibu/code/keyness_g2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()   # 실행하면 파일 선택창이 뜸 → 13개 한꺼번에 선택

Saving 03.txt to 03.txt
Saving 04.txt to 04.txt
Saving 05.txt to 05.txt
Saving 06.txt to 06.txt
Saving 07.txt to 07.txt
Saving 08.txt to 08.txt
Saving 09.txt to 09.txt
Saving 11.txt to 11.txt
Saving 12.txt to 12.txt
Saving 13.txt to 13.txt
Saving 02.txt to 02.txt
Saving 01.txt to 01.txt
Saving 10.txt to 10.txt


In [4]:
import re
from collections import Counter

# 한자 추출 정규식: 기본한자 + 확장A/B
#   확장B 포함 → 𩯽(U+29BFD) 같은 희귀자도 누락 없이 집계
HAN = re.compile(
    r'[\u3400-\u4DBF\u4E00-\u9FFF\U00020000-\U0002A6DF\U0002A700-\U0002EBEF]'
)

# 편집용 표제행 판별 (파일 헤더가 다르면 이 함수만 수정)
def is_header(line):
    return ('笑府' in line and '卷' in line) or ('笑話 본문' in line)

def load_chars(path):
    """텍스트 파일 → 한자 리스트
    - 편집 표제행 제외
    - 한자만 추출 → 편 번호(1,2,…)·문장부호·공백은 자동 제거
    - 篇名(제목)은 본문에 포함 유지
    """
    with open(path, encoding='utf-8') as f:
        body = [ln for ln in f if not is_header(ln)]
    return HAN.findall(''.join(body))

In [5]:
chars = load_chars('10.txt')
print('총 글자수 :', f'{len(chars):,}')
print('자종(고유):', f'{len(set(chars)):,}')
print('상위 10  :', Counter(chars).most_common(10))

총 글자수 : 3,752
자종(고유): 889
상위 10  : [('曰', 144), ('一', 81), ('人', 80), ('子', 77), ('不', 60), ('者', 52), ('鬍', 50), ('有', 49), ('其', 44), ('問', 40)]


In [6]:
# 대상 코퍼스: 形體部 (10번)
target_chars = load_chars('10.txt')

# 참조 코퍼스: 나머지 12부 (10번 제외)
ref_files = ['01.txt','02.txt','03.txt','04.txt','05.txt','06.txt',
             '07.txt','08.txt','09.txt','11.txt','12.txt','13.txt']

# 12개 파일을 차례로 읽어 한 리스트로 이어 붙임
ref_chars = [ch for fp in ref_files for ch in load_chars(fp)]

print(f'形體部  : {len(target_chars):,}자 / 자종 {len(set(target_chars)):,}')
print(f'참조12부: {len(ref_chars):,}자 / 자종 {len(set(ref_chars)):,}')

形體部  : 3,752자 / 자종 889
참조12부: 40,698자 / 자종 2,547


In [8]:
freq_t = Counter(target_chars)   # 形體部 글자별 빈도
freq_r = Counter(ref_chars)      # 참조 글자별 빈도
N_t = sum(freq_t.values())       # 形體部 총 글자수  (= 3,752)
N_r = sum(freq_r.values())       # 참조 총 글자수    (≈ 40,693)

# 빠른 점검: 빈도 합이 글자수와 일치하는지
print('形體部 합계 일치:', sum(freq_t.values()) == len(target_chars))
print('鬍 → 形體部', freq_t['鬍'], '/ 참조', freq_r.get('鬍', 0))

形體部 합계 일치: True
鬍 → 形體部 50 / 참조 1


In [9]:

import math

def log_likelihood(a, b, c, d):
    """2x2 분할표 기반 로그우도비 G²
    (Rayson & Garside 2000; Dunning 1993)
      a = 대상 빈도   b = 참조 빈도
      c = 대상 총자수 d = 참조 총자수
    a=0 또는 b=0인 항은 x·ln x → 0 규약에 따라 합산에서 제외
    """
    E1 = c * (a + b) / (c + d)   # 대상 기대빈도
    E2 = d * (a + b) / (c + d)   # 참조 기대빈도
    g = 0.0
    if a > 0:
        g += a * math.log(a / E1)
    if b > 0:
        g += b * math.log(b / E2)
    return 2 * g

In [10]:
import pandas as pd

rows = []
for ch, a in freq_t.items():               # 形體部의 모든 글자 (필터 없음)
    b = freq_r.get(ch, 0)
    g2 = log_likelihood(a, b, N_t, N_r)
    # 편중 방향: 形體部 상대빈도 > 참조 상대빈도 → '+'(과대표상), 아니면 '−'
    direction = '+' if (a / N_t) > (b / N_r) else '−'
    rows.append({
        '글자': ch,
        '形體部_빈도': a,
        '참조_빈도': b,
        '形體部_상대빈도(‰)': round(a / N_t * 1000, 2),
        '참조_상대빈도(‰)':   round(b / N_r * 1000, 2),
        'G²': round(g2, 2),
        '편중': direction,
    })

df = (pd.DataFrame(rows)
        .sort_values('G²', ascending=False)
        .reset_index(drop=True))
df.insert(0, '순위', df.index + 1)

# 유의 임계값: G² ≥ 15.13 → p<0.0001 (자유도 1, 전지은 2014)
df['유의'] = df['G²'].apply(lambda x: '***' if x >= 15.13 else '')

df.head(30)

,순위,글자,形體部_빈도,참조_빈도,形體部_상대빈도(‰),참조_상대빈도(‰),G²,편중,유의
0,1,鬍,50,1,13.33,0.02,237.54,+,***
1,2,子,77,279,20.52,6.86,58.12,+,***
2,3,聾,8,0,2.13,0.00,39.55,+,***
3,4,瞽,8,0,2.13,0.00,39.55,+,***
4,5,臭,16,15,4.26,0.37,38.81,+,***
5,6,鼻,15,13,4.00,0.32,37.78,+,***
6,7,哉,12,8,3.20,0.20,33.82,+,***
7,8,鬚,10,4,2.67,0.10,33.40,+,***
8,9,矮,6,0,1.60,0.00,29.66,+,***
9,10,辨,9,4,2.40,0.10,29.15,+,***


In [11]:
# 전체 글자 표 저장 (子, 曰 포함 그대로)
df.to_excel('형체부_keyness_전체.xlsx', index=False)              # 엑셀
df.to_csv('형체부_keyness_전체.csv', index=False,
          encoding='utf-8-sig')                                  # CSV(엑셀 한글 호환)

# 코퍼스 규모도 따로 기록 (논문 인용치 확정용)
with open('corpus_size.txt', 'w', encoding='utf-8') as f:
    f.write(f'形體部 : {N_t}자 / 자종 {len(freq_t)}\n')
    f.write(f'참조12부: {N_r}자 / 자종 {len(freq_r)}\n')

print('저장 완료')

저장 완료
